# Section 7 · Observability 4 — LangSmith (Groq edition)

> **Groq edition.** Same lesson as the Bedrock notebook next to this one; only the model setup changes. Needs a `GROQ_API_KEY` (free at [console.groq.com/keys](https://console.groq.com/keys)) in your environment or a `.env` file.

[LangSmith](https://smith.langchain.com) was built for LangChain/LangGraph, but it accepts OpenTelemetry traces from any framework — including Strands.

**Get your key**
1. Sign up at https://smith.langchain.com
2. **Settings → API Keys → Create API Key** (starts with `lsv2_pt_`)

LangSmith receives OTel traces at `https://api.smith.langchain.com/otel`. The API key and project name travel as OTLP headers.

> Each tracing notebook should run in a **fresh kernel**: OpenTelemetry allows only one global tracer provider per Python process.

In [ ]:
%pip install -q -r requirements-groq.txt

In [ ]:
import os

from dotenv import load_dotenv
from strands.models.openai import OpenAIModel

load_dotenv()  # reads GROQ_API_KEY from a .env file, if present

MODEL_ID = os.getenv("GROQ_MODEL_ID", "openai/gpt-oss-120b")

model = OpenAIModel(
    client_args={
        "api_key": os.environ["GROQ_API_KEY"],
        "base_url": os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1"),  # or a local OpenAI-compatible server
        "max_retries": 5,  # retry dropped connections (the client default is 2)
    },
    model_id=MODEL_ID,
    # Groq reasoning models stream their thinking by default; hide it so only the answer prints.
    params={"extra_body": {"include_reasoning": False}},
)
print(f"Using {MODEL_ID} on Groq")

In [ ]:
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY", "lsv2_pt_...")
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT", "strands-agents-demo")

os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = "https://api.smith.langchain.com/otel"
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = f"x-api-key={LANGSMITH_API_KEY},Langsmith-Project={LANGSMITH_PROJECT}"

In [ ]:
from strands import Agent
from strands.telemetry import StrandsTelemetry
from strands_tools import calculator, http_request

strands_telemetry = StrandsTelemetry()
strands_telemetry.setup_otlp_exporter()

agent = Agent(
    model=model,
    system_prompt="You are a helpful research assistant.",
    tools=[http_request, calculator],
)

response = agent("What is 15% of 2,847? Then fetch https://httpbin.org/get and tell me which headers it returns.")
strands_telemetry.tracer_provider.force_flush()

Open your LangSmith project to see the same nested structure: agent invocation → model calls → tool executions, with tokens and timing.

LangSmith also gives you a **Playground** to edit a traced prompt and re-run it, **Datasets** built from production traces for regression tests, and **Annotation queues** for human review.